# 3.2. GPT-2 inference

In this notebook, we will use the GPT-2 model to make sample detoxification. We will use the model trained in the previous notebook.


In [1]:
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
)

In [2]:
KAGGLE_INFER = False

In [3]:
if KAGGLE_INFER:
    model_output_dir = "/kaggle/working/models/gpt2-based"
else:
    model_output_dir = "../models/gpt2-based"

In [4]:
# The format is: [TOX]text[/TOX]»»[DETOX]text[/DETOX]
# [TOX]   - source text
# [DETOX] - target text
# »»      - separator

# So, we will add 5 custom tokens to the vocabulary:
# [TOX], [/TOX], [DETOX], [/DETOX], »»
tokens_dict = {
    "tox_begin": "[TOX]",
    "tox_end": "[/TOX]",
    "detox_begin": "[DETOX]",
    "detox_end": "[/DETOX]",
    "separator": "»»",
    "split": "[SPLIT]",
}

In [5]:
# Load the model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(model_output_dir)
model = GPT2LMHeadModel.from_pretrained(model_output_dir)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [6]:
from transformers import pipeline

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [7]:
# Inference pipeline
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=DEVICE)

In [8]:
def generate(text: str) -> str:
    """
    Generates suggestions for the given text.
    """
    prompt = (
        tokens_dict["tox_begin"]
        + text
        + tokens_dict["tox_end"]
        + tokens_dict["separator"]
        + tokens_dict["detox_begin"]
    )

    return generator(prompt, max_length=len(prompt) * 2.5)[0]["generated_text"]


def get_suggestions(generated_text: str) -> list:
    suggestions = generated_text.split(
        tokens_dict["separator"] + tokens_dict["detox_begin"]
    )[1]

    # Replace all the special tokens with [SPLIT]
    for token in tokens_dict.values():
        suggestions = suggestions.replace(token, tokens_dict["split"])

    # Split by [SPLIT]
    suggestions = suggestions.split(tokens_dict["split"])

    # Trim if any of \", \n, or whitespace is present
    suggestions = [s.strip('"\n ') for s in suggestions]

    # Remove empty strings
    suggestions = list(filter(None, suggestions))

    return suggestions

In [11]:
prompt = "Eat my pussy, punk!"

for suggestion in get_suggestions(generate(prompt)):
    print(suggestion)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


eating my meat, punk!
Eat it, punk!
I'll eat it" I'll eat it
I'm eating it!
I'm eating it!"" It's eating me right now!
I'm eating it!"" He said.
I'm eating it!"" Right now!
I'm cooking it" I'm eating it, punk!
I ate it!
